# MLB Spray Charts for WBC 2026 Roster Players — baseball-field-viz

This notebook visualizes **MLB Statcast data for WBC 2026 roster players** using **[baseball-field-viz](https://github.com/yasumorishima/baseball-field-viz)** — a Python library for drawing baseball fields and spray charts.

> **Note**: This dataset contains 2024–2025 MLB **regular season** Statcast data for players on WBC 2026 rosters. It is **not** WBC game data.

### What baseball-field-viz enables
- `draw_field(ax)` — draws a baseball field on any matplotlib Axes
- `transform_coords(df)` — converts Statcast `hc_x`/`hc_y` to feet (home plate at origin)
- `spraychart(ax, df, color_by='events')` — one-liner spray chart

Unlike pybaseball's built-in `spraychart()`, this library gives direct access to the matplotlib `Axes`, so you can overlay **heatmaps** (seaborn `kdeplot`, `histplot`) on top.

### Dataset
- **WBC 2026 Scouting** — MLB Statcast data (2024–2025) for players on WBC 2026 rosters (18 countries)
- Dataset: [yasunorim/wbc-2026-scouting](https://www.kaggle.com/datasets/yasunorim/wbc-2026-scouting)

### WBC 2026 Scouting Dashboard
An interactive scouting dashboard built from this dataset is available at:  
**https://wbc-2026-scouting-dashboard-zvg.caffeine.xyz/**

---

In [ ]:
!pip install baseball-field-viz -q

import os
import numpy as np
import pandas as pd
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from baseball_field_viz import transform_coords, draw_field, spraychart, draw_strike_zone, pitch_zone_chart

print("baseball-field-viz ready")

In [ ]:
def _find_data_dir():
    candidates = [
        "/kaggle/input/wbc-2026-scouting",
        "/kaggle/input/datasets/yasunorim/wbc-2026-scouting",
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"Data not found. Tried: {candidates}")

DATA_DIR = _find_data_dir()
print(f"Data directory: {DATA_DIR}")

df = pd.read_csv(f"{DATA_DIR}/statcast_batters.csv")
rosters = pd.read_csv(f"{DATA_DIR}/rosters.csv")

print(f"Statcast records : {len(df):,}")
print(f"Countries        : {df['country_name'].nunique()}")
print(f"Players          : {df['player_name'].nunique()}")
print(f"Batted balls     : {df['hc_x'].notna().sum():,}")
print()

hit_events = ['home_run', 'double', 'triple', 'single']
hits_by_country = (
    df[df['events'].isin(hit_events)]
    .groupby('country_name')
    .size()
    .sort_values(ascending=False)
)
print("Hits by country:")
print(hits_by_country.to_string())

## 1. All WBC 2026 Countries — Spray Chart Overview

Hits (single / double / triple / home run) from all 18 WBC 2026 countries, colored by country.  
Data is from the **2024–2025 MLB regular season** for players on each country's WBC roster.

`draw_field(ax)` draws the field, then we manually scatter each country with its own color.

In [ ]:
hits = df[
    df['hc_x'].notna() & df['events'].isin(hit_events)
].copy()
hits = transform_coords(hits)

country_list = sorted(hits['country_name'].unique())
colors = cm.tab20(np.linspace(0, 1, len(country_list)))
color_map = dict(zip(country_list, colors))

fig, ax = plt.subplots(figsize=(12, 12))
draw_field(ax)

for country in country_list:
    subset = hits[hits['country_name'] == country]
    ax.scatter(
        subset['x'], subset['y'],
        c=[color_map[country]], alpha=0.35, s=12,
        label=f"{country} ({len(subset)})"
    )

ax.set_xlim(-350, 350)
ax.set_ylim(-50, 420)
ax.set_xlabel("X (feet)")
ax.set_ylabel("Y (feet)")
ax.set_title("WBC 2026 — All Countries Batted Balls (Hits Only)", fontsize=14)
ax.legend(loc='upper right', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

USA, Dominican Republic, and Venezuela account for the largest share of hits — reflecting the high number of MLB players on those rosters. Despite differing sample sizes, batted ball distributions tend to follow similar patterns across countries, concentrating in center and right-center field.

## 2. Top 4 Countries — Batted Ball Density Comparison

Batted ball density (all contact events) for the four countries with the most MLB players on their WBC 2026 rosters.  
`draw_field(ax)` + seaborn `kdeplot` shows where contact tends to cluster across the field.

In [ ]:
top_countries = ['USA', 'Dominican Republic', 'Venezuela', 'Japan']

fig, axs = plt.subplots(2, 2, figsize=(16, 14))

for ax, country in zip(axs.flat, top_countries):
    df_c = df[df['country_name'] == country]
    df_ct = transform_coords(df_c[df_c['hc_x'].notna()])

    draw_field(ax)
    sns.kdeplot(
        data=df_ct, x='x', y='y', ax=ax,
        fill=True, alpha=0.6, cmap='Reds', levels=8,
        clip=((-350, 350), (0, 400)),
        bw_adjust=0.7,
        thresh=0.1,
    )
    ax.set_xlim(-350, 350)
    ax.set_ylim(-50, 400)
    ax.set_title(f"{country}  (n={len(df_ct):,})", fontsize=13)

plt.suptitle("WBC 2026 — Batted Ball Density by Country", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

All four countries show a similar concentration in center and right-center field, consistent with pull tendencies among MLB hitters. The density map makes it easier to identify zone-level patterns compared to individual scatter points — USA and Dominican Republic show particularly strong clustering toward the pull side, while Japan's distribution is slightly more spread across center field.

## 3. Japan — Hits vs Outs Heatmap

This is where `baseball-field-viz` shines over pybaseball's built-in `spraychart()`:  
since we have direct access to the `Axes` object, we can overlay seaborn's `kdeplot` to show **density distributions**.

- **Left**: Hits heatmap (single / double / triple / home run)
- **Right**: Outs heatmap (all other batted balls)

In [ ]:
df_jpn = df[df['country_name'] == 'Japan']
df_jpn_t = transform_coords(df_jpn[df_jpn['hc_x'].notna()])

hits_jpn = df_jpn_t[df_jpn_t['events'].isin(hit_events)]
outs_jpn = df_jpn_t[~df_jpn_t['events'].isin(hit_events)]

fig, axs = plt.subplots(1, 2, figsize=(16, 8))

draw_field(axs[0])
sns.kdeplot(
    data=hits_jpn, x='x', y='y', ax=axs[0],
    cmap='Reds', fill=True, alpha=0.6, levels=8,
    clip=((-350, 350), (0, 400)), bw_adjust=0.7, thresh=0.1,
)
axs[0].set_xlim(-350, 350)
axs[0].set_ylim(-50, 400)
axs[0].set_title(f"Japan — Hits Heatmap  (n={len(hits_jpn)})", fontsize=13)

draw_field(axs[1])
sns.kdeplot(
    data=outs_jpn, x='x', y='y', ax=axs[1],
    cmap='Blues', fill=True, alpha=0.6, levels=8,
    clip=((-350, 350), (0, 400)), bw_adjust=0.7, thresh=0.1,
)
axs[1].set_xlim(-350, 350)
axs[1].set_ylim(-50, 400)
axs[1].set_title(f"Japan — Outs Heatmap  (n={len(outs_jpn)})", fontsize=13)

plt.suptitle("Japan — Batted Ball Distribution", fontsize=15)
plt.tight_layout()
plt.show()

The heatmap reveals where Japan's batters tend to make contact. Outs are concentrated more in the infield and shallow outfield, while hits spread further. Comparing the two panels helps identify zones where Japan's batters are productive — the kind of insight that's difficult to see from a plain scatter plot.

## 4. Home Run Distribution — Top 5 Countries

Home runs only, for the five countries with the most HRs in the dataset.

In [ ]:
hr_counts = (
    df[df['events'] == 'home_run']
    .groupby('country_name')
    .size()
    .sort_values(ascending=False)
    .head(5)
)
top5_hr_countries = hr_counts.index.tolist()

fig, axs = plt.subplots(1, 5, figsize=(28, 7))

for ax, country in zip(axs, top5_hr_countries):
    df_c = df[(df['country_name'] == country) & (df['events'] == 'home_run')]
    n = len(df_c[df_c['hc_x'].notna()])
    spraychart(ax, df_c, color_by='events', title=f"{country}\n({n} HR)")
    # remove legend for compact display
    ax.get_legend().remove()

plt.suptitle("WBC 2026 — Home Run Spray Charts (Top 5 Countries)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Home run patterns are broadly similar across countries — most land in center to left-center field — reflecting the pull tendencies common among power hitters in MLB. Individual country tendencies are worth exploring, but sample size differences make direct comparisons tentative.

## 5. Strike Zone Drawing — v0.2.1 New Feature

baseball-field-viz v0.2.1 adds two new functions:

| Function | Description |
|---|---|
| `draw_strike_zone(ax, sz_top, sz_bot)` | Draw strike zone rectangle in plate_x/plate_z coordinates |
| `pitch_zone_chart(ax, df, color_by)` | Plot pitch location **density (kdeplot)** with auto-sized strike zone |

`pitch_zone_chart` uses seaborn `kdeplot` (filled contours) by default — showing where pitches cluster rather than individual dots.  
The strike zone is defined in `plate_x` (horizontal, feet) and `plate_z` (vertical, feet from ground).  
Home plate width is 17 inches = 1.4167 feet, centered at `plate_x = 0`.

In [ ]:
# --- draw_strike_zone: visualize different strike zone sizes ---
fig, axs = plt.subplots(1, 2, figsize=(12, 6))

for ax, (sz_t, sz_b, label) in zip(axs, [
    (3.5, 1.5, "Average MLB Strike Zone"),
    (4.0, 1.3, "Taller Batter (e.g. 6'7\")"),
]):
    draw_strike_zone(ax, sz_top=sz_t, sz_bot=sz_b)
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(0, 5.5)
    ax.set_aspect('equal')
    ax.set_xlabel("plate_x (ft)")
    ax.set_ylabel("plate_z (ft)")
    ax.set_title(f"{label}\n(sz_top={sz_t}, sz_bot={sz_b})")

plt.suptitle("draw_strike_zone() — baseball-field-viz v0.2.0", fontsize=14)
plt.tight_layout()
plt.show()

# --- pitch_zone_chart: pitch locations from statcast_pitchers ---
df_p = pd.read_csv(f"{DATA_DIR}/statcast_pitchers.csv")
jpn_p = df_p[df_p['country_name'] == 'Japan']
print(f"Japan pitcher records: {len(jpn_p):,}")

if 'plate_x' in jpn_p.columns and 'plate_z' in jpn_p.columns:
    top2 = jpn_p['player_name'].value_counts().head(2).index.tolist()
    fig, axs = plt.subplots(1, len(top2), figsize=(7 * len(top2), 7))
    if len(top2) == 1:
        axs = [axs]
    for ax, name in zip(axs, top2):
        df_pt = jpn_p[jpn_p['player_name'] == name]
        pitch_zone_chart(ax, df_pt, color_by='pitch_type',
                         sz_top=3.5, sz_bot=1.5, title=name)
    plt.suptitle("Japan — Pitch Locations (pitch_zone_chart)", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("Note: plate_x/plate_z not in this dataset.")
    print("Use pybaseball.statcast_pitcher() for full pitch location data.")

The strike zone in Statcast is measured per-pitch by Hawk-Eye (`sz_top`/`sz_bot` columns), not derived from player height — so it reflects each batter's actual stance. When present in the DataFrame, `pitch_zone_chart` uses their mean automatically. Use `pybaseball.statcast()` for full Statcast data including pitch-level columns.

## Explore More: WBC 2026 Scouting Dashboard

For an interactive view of each player's MLB stats by country, check out the **WBC 2026 Scouting Dashboard**:

**https://wbc-2026-scouting-dashboard-zvg.caffeine.xyz/**

The dashboard shows per-player batting and pitching metrics based on the same 2024–2025 MLB Statcast data for WBC 2026 roster players.

---

## Resources

| | Link |
|---|---|
| baseball-field-viz (PyPI) | https://pypi.org/project/baseball-field-viz/ |
| baseball-field-viz (GitHub) | https://github.com/yasumorishima/baseball-field-viz |
| WBC 2026 Scouting Dataset | https://www.kaggle.com/datasets/yasunorim/wbc-2026-scouting |
| WBC 2026 Scouting Dashboard | https://wbc-2026-scouting-dashboard-zvg.caffeine.xyz/ |

```python
pip install baseball-field-viz
```

```python
from baseball_field_viz import draw_field, spraychart, transform_coords

fig, ax = plt.subplots(figsize=(10, 10))
spraychart(ax, df, color_by='events')
plt.show()
```